In [ ]:
# # This Python 3 environment comes with many helpful analytics libraries installed
# # It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# # For example, here's several helpful packages to load

# import numpy as np # linear algebra
# import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# # Input data files are available in the read-only "../input/" directory
# # For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

# import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# # You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# # You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# # Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# # Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

# import kagglehub
# # kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
# import pandas as pd
# data = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv")
# data.to_csv("submission.csv", index=False)

**Import modules**

In [ ]:
import numpy as np, pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from collections import Counter

In [ ]:
TRAIN_PATH  = "/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv"
TEST_PATH   = "/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv"
SAMPLE_PATH = "/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv"
OUTPUT_PATH = "/kaggle/working/submission.csv"
 
LABELS=["A","B","C","D","E"]; L2I={l:i for i,l in enumerate(LABELS)}

MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"
FALLBACK_MODEL = "Qwen/Qwen2.5-3B-Instruct"
DUP_SIM_THRESHOLD = 0.92  

In [ ]:
train_df=pd.read_csv(TRAIN_PATH); test_df=pd.read_csv(TEST_PATH); sample=pd.read_csv(SAMPLE_PATH)
ANSWER_COL="answer" if "answer" in train_df.columns else train_df.columns[-1]
print("Train",train_df.shape,"Test",test_df.shape,"| answer col:",ANSWER_COL)

def full_text(r): return str(r["prompt"])+" "+" ".join(str(r[c]) for c in LABELS)
def heuristic_scores(row):
    options=[str(row[c]) for c in LABELS]
    lens=np.array([len(o) for o in options],float); len_s=lens/(lens.sum()+1e-9)
    wcnt=np.array([len(o.split()) for o in options],float); wc_s=wcnt/(wcnt.sum()+1e-9)
    try:
        v=TfidfVectorizer(ngram_range=(1,2),sublinear_tf=True).fit_transform([str(row['prompt'])]+options)
        ts=cosine_similarity(v[0:1],v[1:])[0]; tf_s=ts/(ts.sum()+1e-9)
    except Exception: tf_s=np.full(5,0.2)
    s=0.85*len_s+0.03*tf_s+0.12*wc_s if np.std(lens)>=20 else 0.60*len_s+0.30*tf_s+0.10*wc_s
    return s

In [ ]:

tr_txt=[full_text(r) for _,r in train_df.iterrows()]; te_txt=[full_text(r) for _,r in test_df.iterrows()]
wv=TfidfVectorizer(ngram_range=(1,2),sublinear_tf=True,max_features=200000)
cz=TfidfVectorizer(analyzer="char_wb",ngram_range=(3,5),sublinear_tf=True,max_features=120000)
wv.fit(tr_txt+te_txt); cz.fit(tr_txt+te_txt)
Xw_tr=wv.transform(tr_txt); Xc_tr=cz.transform(tr_txt); Xw_te=wv.transform(te_txt); Xc_te=cz.transform(te_txt)
sim_w=cosine_similarity(Xw_te,Xw_tr); sim_c=0.6*sim_w+0.4*cosine_similarity(Xc_te,Xc_tr)
tli=np.array([L2I[a] for a in train_df[ANSWER_COL]])

def retrieval_vote(i):
    order=np.argsort(sim_w[i])[::-1][:30]; votes=np.zeros(5); tw=0.0
    for j in order:
        s=sim_w[i][j]
        if s<0.80: break
        votes[tli[j]]+=s**3; tw+=s**3
    top_sim=float(sim_c[i].max())
    return (votes if tw>0 else None), top_sim

n_dup=sum(1 for i in range(len(test_df)) if sim_c[i].max()>=DUP_SIM_THRESHOLD)
print(f"Estimated memorized duplicates in test: {n_dup}/{len(test_df)} ({n_dup/len(test_df):.0%})")
print("These are handled by retrieval; the LLM focuses on the rest.")

In [ ]:
USE_LLM=True
try:
    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
    if not torch.cuda.is_available():
        print("No GPU detected -> falling back to retrieval+heuristic only."); USE_LLM=False
except Exception as e:
    print("transformers/torch unavailable -> retrieval+heuristic only:", e); USE_LLM=False

tokenizer=model=None
if USE_LLM:
    def load(name):
        tok=AutoTokenizer.from_pretrained(name)
        bnb=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16,
                               bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True)
        mdl=AutoModelForCausalLM.from_pretrained(name, quantization_config=bnb,
                                                 device_map="auto", torch_dtype=torch.float16)
        mdl.eval(); return tok,mdl
    try:
        tokenizer,model=load(MODEL_NAME); print("Loaded",MODEL_NAME)
    except Exception as e:
        print("Primary model failed (",e,") -> trying fallback",FALLBACK_MODEL)
        try: tokenizer,model=load(FALLBACK_MODEL); print("Loaded",FALLBACK_MODEL)
        except Exception as e2: print("Fallback failed:",e2,"-> retrieval only."); USE_LLM=False


letter_ids=None
if USE_LLM:
    letter_ids={}
    for L in LABELS:
        ids=set()
        for s in (L, " "+L):
            t=tokenizer.encode(s, add_special_tokens=False)
            if len(t)==1: ids.add(t[0])
        letter_ids[L]=list(ids) if ids else tokenizer.encode(L,add_special_tokens=False)[:1]

def llm_scores(row):
    
    q=str(row["prompt"])
    opts="\n".join(f"{c}. {row[c]}" for c in LABELS)
    user=(f"Answer the multiple choice question. Reply with only the letter of the best option.\n\n"
          f"Question: {q}\n\nOptions:\n{opts}\n\nAnswer:")
    msgs=[{"role":"user","content":user}]
    text=tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    enc=tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        logits=model(**enc).logits[0,-1,:].float()
    sc=np.array([max(float(logits[i]) for i in letter_ids[L]) for L in LABELS])
    sc=sc-sc.max(); p=np.exp(sc); return p/p.sum()

In [ ]:
from tqdm.auto import tqdm
preds=[]
for i in tqdm(range(len(test_df)), desc="Predicting"):
    row=test_df.iloc[i]
    votes, top_sim = retrieval_vote(i)
    heur = heuristic_scores(row)
    lm = llm_scores(row) if USE_LLM else None

    if votes is not None and top_sim>=DUP_SIM_THRESHOLD:
        # memorized duplicate: trust retrieved letter for rank-1
        r1=int(np.argmax(votes))
        sec = (lm if lm is not None else heur).copy(); sec[r1]=-1e9
        r2=int(np.argmax(sec)); sec[r2]=-1e9; r3=int(np.argmax(sec))
    elif lm is not None:
        # novel question: LLM answers; nudge with retrieval votes if a weak match exists
        score = lm.copy()
        if votes is not None:
            score = 0.7*lm + 0.3*(votes/votes.sum())
        order=np.argsort(score)[::-1][:3]; r1,r2,r3=[int(x) for x in order]
    else:
        # no LLM available -> retrieval+heuristic (reproduces ~0.758)
        if votes is not None:
            r1=int(np.argmax(votes)); h=heur.copy(); h[r1]=-1; r2=int(np.argmax(h)); h[r2]=-1; r3=int(np.argmax(h))
        else:
            r1,r2,r3=[int(x) for x in np.argsort(heur)[::-1][:3]]
    preds.append([r1,r2,r3])

pred_str=[" ".join(LABELS[i] for i in p) for p in preds]

In [ ]:
id_col,pred_col=sample.columns[0],sample.columns[1]
submission=pd.DataFrame({id_col:test_df["id"].values, pred_col:pred_str})
try: submission=sample[[id_col]].merge(submission,on=id_col,how="left")
except Exception: pass
submission.to_csv(OUTPUT_PATH,index=False)
allp=[p.split() for p in submission[pred_col]]
print("Rank-1 dist:",dict(Counter(p[0] for p in allp)))
print(f"Saved {OUTPUT_PATH} | rows {len(submission)} | LLM used: {USE_LLM}")
print(submission.head(10).to_string(index=False))

In [ ]:
# train_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
# test_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")


In [ ]:
# print(train_df.shape)
# print(test_df.shape)

In [ ]:
# print(train_df.columns.tolist())
# print(test_df.columns.tolist())